# COBYQA Sequential Optimization

This notebook demonstrates how to use the COBYQA (Constrained Optimization BY Quadratic Approximation) algorithm in Xopt for sequential derivative-free optimization.

## Overview

COBYQA is a trust-region derivative-free optimization method that:
- Uses local quadratic models to guide the search
- Automatically handles variable bounds
- Works well for expensive black-box function evaluations
- Suitable for single-objective optimization problems

In Xopt, COBYQAGenerator implements the ask/tell pattern, allowing you to:
1. Request the next candidate point to evaluate
2. Evaluate it externally (e.g., run a simulation)
3. Provide the result back to the optimizer
4. Repeat until convergence

## Setup

In [1]:
import math
import numpy as np
import pandas as pd
from xopt import Xopt
from xopt.vocs import VOCS
from xopt.evaluator import Evaluator
from xopt.generators.sequential.cobyqa import COBYQAGenerator

## Example 1: Simple 2D Rosenbrock Function

Let's optimize the Rosenbrock function, a classic test case for optimization algorithms.

In [2]:
def rosenbrock(input_dict):
    """Rosenbrock function: f(x,y) = (1-x)^2 + 100(y-x^2)^2"""
    x = input_dict["x"]
    y = input_dict["y"]
    return {"f": (1 - x) ** 2 + 100 * (y - x ** 2) ** 2}

# Test the function
print(f"f(0, 0) = {rosenbrock({'x': 0, 'y': 0})['f']:.2f}")
print(f"f(1, 1) = {rosenbrock({'x': 1, 'y': 1})['f']:.2f}")

f(0, 0) = 1.00
f(1, 1) = 0.00


### Define the optimization problem

In [8]:
# Define variables and objective
vocs = VOCS(
    variables={"x": [-2, 2], "y": [-1, 3]},
    objectives={"f": "MINIMIZE"},
)

# Create evaluator
evaluator = Evaluator(function=rosenbrock)

# Create COBYQA generator with initial point
generator = COBYQAGenerator(
    vocs=vocs,
    initial_point={"x": 0.0, "y": 0.0},
    options={"maxiter": 100},  # scipy COBYQA options
)

# Create Xopt object
X = Xopt(evaluator=evaluator, generator=generator)

### Run optimization

In [9]:
# Run optimization for 50 steps
for i in range(50):
    X.step()
    if (i + 1) % 5 == 0:
        best_idx = X.data["f"].argmin()
        best_f = X.data["f"].iloc[best_idx]
        print(f"Step {i + 1:2d}: f = {best_f:.6f}")

print("\nOptimization complete!")

Step  5: f = 1.000000
Step 10: f = 0.703637
Step 15: f = 0.532265
Step 20: f = 0.390269
Step 25: f = 0.239811
Step 30: f = 0.181896
Step 35: f = 0.103104
Step 40: f = 0.101353
Step 45: f = 0.082039
Step 50: f = 0.049387

Optimization complete!


### View results

In [5]:
# Show final result
best_idx = X.data["f"].argmin()
best_point = X.data.iloc[best_idx]

print(f"Number of evaluations: {len(X.data)}")
print(f"\nBest result:")
print(f"  x = {best_point['x']:.6f}")
print(f"  y = {best_point['y']:.6f}")
print(f"  f = {best_point['f']:.6f}")
print(f"\nOptimal point: (1, 1) with f = 0")

Number of evaluations: 20

Best result:
  x = 0.386789
  y = 0.137672
  f = 0.390269

Optimal point: (1, 1) with f = 0


## Example 2: Using YAML Configuration

You can also configure COBYQA using YAML strings for easier serialization and reproducibility.

In [6]:
yaml_config = """
generator:
  name: cobyqa
  initial_point: {x: -1.5, y: 2.5}
  options:
    maxiter: 150
  vocs:
    variables:
      x: [-2, 2]
      y: [-1, 3]
    objectives:
      f: MINIMIZE

evaluator:
  function: __main__.rosenbrock
"""

# Note: In a real notebook, you'd use the actual module path
# For this example, we'll use the direct Python approach above instead
print("YAML configuration ready (see above)")

YAML configuration ready (see above)


## Example 3: Sphere Function (Simpler Case)

Let's also try a simpler sphere function to see faster convergence.

In [7]:
def sphere_3d(input_dict):
    """Sphere function in 3D: f(x,y,z) = x^2 + y^2 + z^2"""
    x = input_dict["x"]
    y = input_dict["y"]
    z = input_dict["z"]
    return {"f": x**2 + y**2 + z**2}

# Define the problem
vocs_3d = VOCS(
    variables={
        "x": [-5, 5],
        "y": [-5, 5],
        "z": [-5, 5],
    },
    objectives={"f": "MINIMIZE"},
)

# Create Xopt with COBYQA
evaluator_3d = Evaluator(function=sphere_3d)
generator_3d = COBYQAGenerator(
    vocs=vocs_3d,
    initial_point={"x": 3.0, "y": 3.0, "z": 3.0},
)

X_3d = Xopt(evaluator=evaluator_3d, generator=generator_3d)

# Run optimization
for i in range(15):
    X_3d.step()

# Show results
best_idx = X_3d.data["f"].argmin()
best_point_3d = X_3d.data.iloc[best_idx]

print(f"3D Sphere Optimization Results:")
print(f"  Evaluations: {len(X_3d.data)}")
print(f"  Best x = {best_point_3d['x']:.6f}")
print(f"  Best y = {best_point_3d['y']:.6f}")
print(f"  Best z = {best_point_3d['z']:.6f}")
print(f"  Best f = {best_point_3d['f']:.6f}")

3D Sphere Optimization Results:
  Evaluations: 15
  Best x = 0.117773
  Best y = 0.176660
  Best z = 0.176660
  Best f = 0.076288


## Key Parameters

- **`initial_point`**: Starting point for the optimization (required for COBYQAGenerator)
- **`tol`**: Termination tolerance (passed to scipy.optimize.minimize)
- **`options`**: Dictionary of scipy COBYQA options:
  - `maxiter`: Maximum number of iterations
  - `rhobeg`: Initial trust region radius
  - `rhoend`: Final trust region radius
  - Other scipy COBYQA options as documented in scipy

## Advantages of COBYQA

1. **Derivative-free**: No need for gradient information
2. **Trust-region based**: Efficient use of function evaluations
3. **Bounds handling**: Native support for variable bounds
4. **Quadratic models**: Uses quadratic approximations for guidance
5. **Ask/tell pattern**: Works naturally with external evaluators

## When to Use COBYQA

- Single-objective optimization problems
- Expensive function evaluations (simulation, experiment, etc.)
- No gradient information available
- Moderate dimensionality (typically < 100 variables)
- Deterministic or low-noise evaluation functions